In [ ]:
%autoreload 2

In [1]:
%reload_ext autoreload
import os, sys, random
import numpy as np
import pandas as pd
import seaborn as sns

import matplotlib.pyplot as plt
from matplotlib import  rcParams
from fish import Gafftopsail
sys.path.append(r'C://Users//Zichen//anaconda3//envs//2ptank//Lib//site-packages//')
import utils, barcode
path = (r'C:/Data/Imaging\260425_overlap//fish5//')
rcParams['font.size'] = 12

In [5]:
fish = Gafftopsail(path, filelist = ['stimulus', 'imaging', 'alignment', 'processed_tail', 'processed_eye'], sequence = 5)
fish.stimulus_df.stim_name = [str(s) for s in fish.stimulus_df.stim_name]

low passs assuming framerate ~2
smoothed
low passs assuming framerate ~2
smoothed
low passs assuming framerate ~2
smoothed
low passs assuming framerate ~2
smoothed
low passs assuming framerate ~2
smoothed
eye not proofread


__EACH STIM__

In [6]:
#axis 0: trial; axis 1: neuron; axis 2: frame
#from stationary start to duration + 20
f_pertrial_dict = barcode.get_pertrial_f(fish)

getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s


In [8]:
fish.saccade_df

,cont_tuples_eyeindex,cont_tuples_realtime_s,e0_change,e1_change,eye_stimuli,eye_realstimuli,eye_fromstart_s,e_conv_change,e_conj_change
0,"(17, 56)","(-7.377272, -5.578081)",25.594434,37.499074,NaN,NaN,NaN,11.904639,63.093508
1,"(486, 506)","(16.38718, 17.38551)",-6.680571,-13.825866,right,stationary,16.387180,-7.145294,-20.506437
2,"(748, 763)","(29.71155, 30.478497)",-4.107835,-5.192492,right,right,29.711550,-1.084657,-9.300327
3,"(810, 828)","(32.788321, 33.698886)",5.958621,6.662689,right,right,32.788321,0.704068,12.621311
4,"(1644, 1676)","(75.450237, 77.017047)",12.173712,18.177125,left,stationary,5.351688,6.003413,30.350837
...,...,...,...,...,...,...,...,...,...
261,"(76041, 76074)","(3841.707846, 3843.26169)",16.236270,23.922282,"['dot_l', 'left']",stationary,4.734340,7.686012,40.158552
262,"(76451, 76470)","(3861.385226, 3862.283822)",4.559692,13.007201,"['dot_l', 'left']",left,24.411720,8.447509,17.566893
263,"(77072, 77099)","(3891.311201, 3892.587786)",8.715014,13.715370,"['dot_l', 'backward']",stationary,19.282438,5.000357,22.430384
264,"(77249, 77278)","(3899.729689, 3901.115982)",-5.590771,14.833044,"['dot_l', 'backward']","['dot_l', 'backward']",27.700926,20.423816,9.242273


__Calculate the DSI of the neuron__

In [ ]:
dot_stim = None
stim_responses = barcode.get_grating_dsi(fish,f_pertrial_dict,baseline_s=7,response_s=15, dot_stim = dot_stim)

__barcoding 2: large response__

In [ ]:
barred_gratingneurons = barcode.select_gratingbarcode(fish, f_pertrial_dict, baseline_s = 7, response_s =15, perc_trial_threshold = 1)

In [ ]:
barred_gratingneurons = barcode.sort_gratingneurons(fish, f_pertrial_dict, barred_gratingneurons)

In [ ]:
# fig = barcode.plot_gratingneurons(fish, f_pertrial_dict, barred_gratingneurons, stim_responses = stim_responses)
# plt.savefig(fish.path + f'//Graphs//grating_barcode.png')
# plt.show()
# plt.close()

__rainbow plot__

In [ ]:
#combine barcoding and stim_responses
for stim, neurons in barred_gratingneurons.items():
    stim_responses[f'barred_{stim}'] = stim_responses.index.isin(neurons)

#save it
os.makedirs(os.path.join(fish.path, "analyze_results"), exist_ok=True)
if dot_stim is not None:
    stim_responses.to_csv(fish.path + f'//analyze_results//{dot_stim}grating_responses.csv')
else:
    stim_responses.to_csv(fish.path + f'//analyze_results//grating_responses.csv')

In [ ]:
barcode.plot_rainbow(fish, stim_responses)
plt.savefig(fish.path + f'//Graphs//grating_rainbow.png')
plt.show()
plt.close()

__Look at response during overlap__

In [1]:
def get_f(fish, highDSI, barred_gratingneurons):
    """Get the avg trace of a population of cells"""
    #look at vis responsive cells
    region_list = [['prosencephalon_(forebrain)', 'mesencephalon_(midbrain)'], 'rhombencephalon_(hindbrain)', 'tectum', 'pretectum']

    #get traces
    f_dict = {'forebrain':{}, 'hindbrain':{}, 'tectum': {}, 'pretectum': {}}

    fig, ax = plt.subplots(len(barred_gratingneurons), 1 + 2 * len(region_list), figsize = (20, 20))
    #each direction: line vs dot line
    for s, stim in enumerate(barred_gratingneurons):
        stim_n = set(barred_gratingneurons[stim])
        #DSI selection
        if highDSI:
            DSI_boundary = 0.5
            angle_boundary = 45
        else:
            DSI_boundary = 0
            angle_boundary = 1000
        dsi_n = set(stim_responses[(stim_responses['DSI2'] > DSI_boundary) &
                                   (np.abs((stim_responses['dir'] - utils.omr_angles[stim] + 180) % 360 - 180) <= angle_boundary)].index)
        print(f"DSI > {DSI_boundary}, angle within +/-{angle_boundary}deg")
        ax_imshow = ax[s, 0]
        ax_imshow.imshow(fish.img_dict[2], origin = 'lower')
        ax_imshow.set_axis_off()
        for r, (region, region_c) in enumerate(zip(region_list, ['white', 'grey', 'yellow','green'])):
            ax_line = ax[s, r * 2 + 1]
            ax_heat = ax[s, r * 2 + 2]
            f_df = []
            if type(region) == list:
                region_n = set(fish.apos_all.index[fish.apos_all['regions'].apply(lambda x: any(ri in x for ri in region))])#set(fish.apos_all.index)#
            else:
                region_n = set(fish.apos_all.index[fish.apos_all['regions'].apply(lambda x: region in x)])
            region_n = list(region_n & stim_n & dsi_n)

            opp_stim = utils.angles_omr[(utils.omr_angles[stim] + 180)%360]
            for g, (grating, color) in enumerate(zip([stim, opp_stim], [utils.omr_colors[stim], utils.omr_colors[opp_stim]])):
                for d, (dot, linestyle) in enumerate(zip([None, 'dot_l', 'dot_r'], ['-', ':', ':'])):
                    if dot == None:
                        real_stim = grating
                    else:
                        real_stim =  str([dot, grating])

                    #get fluorscnece
                    f_acrosstrial = f_pertrial_dict[real_stim][:, region_n].mean(axis = 0)
                    f_mean = f_acrosstrial.mean(axis = 0)
                    f_sem = f_acrosstrial.std(axis=0) / np.sqrt(f_acrosstrial.shape[0])
                    df = pd.DataFrame(f_acrosstrial)
                    if len(df) > 0:
                        df.loc[:, 'barred'] = stim
                        df.loc[:, 'stim'] = real_stim
                    f_df.append(df)
                    #start plotting
                    ax_imshow.scatter(fish.pos_all.loc[region_n, 'xpos'], fish.pos_all.loc[region_n, 'ypos'], marker = ',', s = 1, color = region_c)

                    ax_line.plot(f_mean, c=color, linestyle = linestyle)
                    ax_line.fill_between(np.arange(len(f_mean)),f_mean - f_sem, f_mean + f_sem,color=color,alpha=0.3, linewidth = 0)
                    ax_line.set_ylim([0.1, 0.6])

                    ax_heat.imshow(f_acrosstrial, aspect='auto',extent=[0, len(f_mean), g * 3 + d, g * 3 + d+1],cmap='viridis', vmin = 0.2, vmax = 0.6)
                    ax_heat.axhline(g * 3 + d, linestyle='--', color='white')
            ax_heat.set_ylim([0, g * 3 + d + 1])
            if s == 0:
                f_dict[list(f_dict.keys())[r]] = pd.concat(f_df, axis = 0, ignore_index = True)
            else:
                f_dict[list(f_dict.keys())[r]] = pd.concat([f_dict[list(f_dict.keys())[r]], pd.concat(f_df, axis = 0, ignore_index = True)], axis = 0, ignore_index = True)
    return f_dict

In [ ]:
highDSI = False
grating_opp = None#'blah'
f_dict = get_f(fish, highDSI)
plt.show()
plt.close()

In [3]:
#combine across fish
paths =[
 r"C:\Data\Imaging\260415_overlap\fish3",r"C:\Data\Imaging\260415_overlap\fish3_2",
r"C:\Data\Imaging\260415_overlap\fish4", r"C:\Data\Imaging\260415_overlap\fish4_2",
     r"C:\Data\Imaging\260415_overlap\fish6", r"C:\Data\Imaging\260415_overlap\fish6_2",
r"C:\Data\Imaging\260425_overlap\fish1", r"C:\Data\Imaging\260425_overlap\fish1_2",
r"C:\Data\Imaging\260425_overlap\fish2",r"C:\Data\Imaging\260425_overlap\fish2_2",
r"C:\Data\Imaging\260425_overlap\fish3",r"C:\Data\Imaging\260425_overlap\fish3_2",
r"C:\Data\Imaging\260425_overlap\fish5",r"C:\Data\Imaging\260425_overlap\fish5_2",
 r"C:\Data\Imaging\260425_overlap\fish6", r"C:\Data\Imaging\260425_overlap\fish6_2",
r"C:\Data\Imaging\260425_overlap\fish7",r"C:\Data\Imaging\260425_overlap\fish7_2",
r"C:\Data\Imaging\260425_overlap\fish8",r"C:\Data\Imaging\260425_overlap\fish8_2",
r"C:\Data\Imaging\260425_overlap\fish9", r"C:\Data\Imaging\260425_overlap\fish9_2",
 r"C:\Data\Imaging\260425_overlap\fish10", r"C:\Data\Imaging\260425_overlap\fish10_2",
r"C:\Data\Imaging\260425_overlap\fish11", r"C:\Data\Imaging\260425_overlap\fish11_2",]
for path in paths:
    fish = Gafftopsail(path + '//', filelist=['stimulus', 'imaging', 'alignment'], sequence=5)
    fish.stimulus_df.stim_name = [str(s) for s in fish.stimulus_df.stim_name]
    #axis 0: trial; axis 1: neuron; axis 2: frame
    #from stationary start to duration + 20
    f_pertrial_dict = barcode.get_pertrial_f(fish)
    for dot_stim in [None, 'dot_l', 'dot_r']:
        stim_responses = barcode.get_grating_dsi(fish, f_pertrial_dict, baseline_s=7, response_s=12, dot_stim=dot_stim)
        if dot_stim == None:
            barred_gratingneurons = barcode.select_gratingbarcode(fish, f_pertrial_dict, baseline_s=7, response_s=12, perc_trial_threshold=1)
        # barred_gratingneurons = barcode.sort_gratingneurons(fish, f_pertrial_dict, barred_gratingneurons)

        #combine barcoding and stim_responses
        for stim, neurons in barred_gratingneurons.items():
            stim_responses[f'barred_{stim}'] = stim_responses.index.isin(neurons)

        #save it
        os.makedirs(os.path.join(fish.path, "analyze_results"), exist_ok=True)
        if dot_stim is not None:
            stim_responses.to_csv(fish.path + f'//analyze_results//{dot_stim}grating_responses_delayed.csv')
        else:
            stim_responses.to_csv(fish.path + f'//analyze_results//grating_responses_delayed.csv')

    highDSI = True
    f_dict = get_f(fish, highDSI, barred_gratingneurons)
    for region in ['forebrain', 'hindbrain', 'tectum', 'pretectum']:
                df = f_dict[region]
                df.to_csv(fish.path + f"//analyze_results//grating_{region}neurons_highDSI.csv")

    highDSI = False
    f_dict = get_f(fish, highDSI, barred_gratingneurons)
    for region in ['forebrain', 'hindbrain', 'tectum', 'pretectum']:
                df = f_dict[region]
                df.to_csv(fish.path + f"//analyze_results//grating_{region}neurons.csv")

        plt.close()

low passs assuming framerate ~2
smoothed
low passs assuming framerate ~2
smoothed
low passs assuming framerate ~2
smoothed
low passs assuming framerate ~2
smoothed
low passs assuming framerate ~2
smoothed
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
baseline: 13 to 20
response: 30 to 32
baseline: 13 to 20
response: 30 to 32
baseline: 13 to 20
response: 30 to 32
baseline: 13 to 20
response: 30 to 32
selection criteria: max >= baseline mean * 2
selection cri

C:\Users\Zichen\AppData\Local\Temp\ipykernel_63248\2529180685.py:46: RuntimeWarning: Mean of empty slice.
  f_mean = f_acrosstrial.mean(axis = 0)
C:\Users\Zichen\anaconda3\envs\2ptank\Lib\site-packages\numpy\_core\_methods.py:137: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
C:\Users\Zichen\anaconda3\envs\2ptank\Lib\site-packages\numpy\_core\_methods.py:223: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\Users\Zichen\anaconda3\envs\2ptank\Lib\site-packages\numpy\_core\_methods.py:181: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
C:\Users\Zichen\anaconda3\envs\2ptank\Lib\site-packages\numpy\_core\_methods.py:212: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
C:\Users\Zichen\AppData\Local\Temp\ipykernel_63248\2529180685.py:46: RuntimeWarning: Mean of empty slice.
  f_mean = f_acrosstrial.mean(axis = 0)


DSI > 0.5, angle within +/-45deg
DSI > 0, angle within +/-1000deg
DSI > 0, angle within +/-1000deg
DSI > 0, angle within +/-1000deg
DSI > 0, angle within +/-1000deg
baseline: 13 to 20
response: 30 to 32
baseline: 13 to 20
response: 30 to 32
baseline: 13 to 20
response: 30 to 32
baseline: 13 to 20
response: 30 to 32
DSI > 0.5, angle within +/-45deg
DSI > 0.5, angle within +/-45deg
DSI > 0.5, angle within +/-45deg


C:\Users\Zichen\AppData\Local\Temp\ipykernel_63248\2529180685.py:46: RuntimeWarning: Mean of empty slice.
  f_mean = f_acrosstrial.mean(axis = 0)
C:\Users\Zichen\anaconda3\envs\2ptank\Lib\site-packages\numpy\_core\_methods.py:137: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
C:\Users\Zichen\anaconda3\envs\2ptank\Lib\site-packages\numpy\_core\_methods.py:223: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\Users\Zichen\anaconda3\envs\2ptank\Lib\site-packages\numpy\_core\_methods.py:181: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
C:\Users\Zichen\anaconda3\envs\2ptank\Lib\site-packages\numpy\_core\_methods.py:212: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
C:\Users\Zichen\AppData\Local\Temp\ipykernel_63248\2529180685.py:46: RuntimeWarning: Mean of empty slice.
  f_mean = f_acrosstrial.mean(axis = 0)


DSI > 0.5, angle within +/-45deg
DSI > 0, angle within +/-1000deg
DSI > 0, angle within +/-1000deg
DSI > 0, angle within +/-1000deg
DSI > 0, angle within +/-1000deg
baseline: 13 to 20
response: 30 to 32
baseline: 13 to 20
response: 30 to 32
baseline: 13 to 20
response: 30 to 32
baseline: 13 to 20
response: 30 to 32
DSI > 0.5, angle within +/-45deg
DSI > 0.5, angle within +/-45deg
DSI > 0.5, angle within +/-45deg


...

DSI > 0, angle within +/-1000deg
DSI > 0, angle within +/-1000deg
DSI > 0, angle within +/-1000deg
DSI > 0, angle within +/-1000deg
baseline: 13 to 20
response: 30 to 32
baseline: 13 to 20
response: 30 to 32
baseline: 13 to 20
response: 30 to 32
baseline: 13 to 20
response: 30 to 32
DSI > 0.5, angle within +/-45deg
DSI > 0.5, angle within +/-45deg
DSI > 0.5, angle within +/-45deg


C:\Users\Zichen\AppData\Local\Temp\ipykernel_63248\2529180685.py:46: RuntimeWarning: Mean of empty slice.
  f_mean = f_acrosstrial.mean(axis = 0)
C:\Users\Zichen\anaconda3\envs\2ptank\Lib\site-packages\numpy\_core\_methods.py:137: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
C:\Users\Zichen\anaconda3\envs\2ptank\Lib\site-packages\numpy\_core\_methods.py:223: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\Users\Zichen\anaconda3\envs\2ptank\Lib\site-packages\numpy\_core\_methods.py:181: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
C:\Users\Zichen\anaconda3\envs\2ptank\Lib\site-packages\numpy\_core\_methods.py:212: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
C:\Users\Zichen\AppData\Local\Temp\ipykernel_63248\2529180685.py:46: RuntimeWarning: Mean of empty slice.
  f_mean = f_acrosstrial.mean(axis = 0)


DSI > 0.5, angle within +/-45deg
DSI > 0, angle within +/-1000deg
DSI > 0, angle within +/-1000deg
DSI > 0, angle within +/-1000deg
DSI > 0, angle within +/-1000deg
baseline: 13 to 20
response: 30 to 32
baseline: 13 to 20
response: 30 to 32
baseline: 13 to 20
response: 30 to 32
baseline: 13 to 20
response: 30 to 32
DSI > 0.5, angle within +/-45deg
DSI > 0.5, angle within +/-45deg
DSI > 0.5, angle within +/-45deg


C:\Users\Zichen\AppData\Local\Temp\ipykernel_63248\2529180685.py:46: RuntimeWarning: Mean of empty slice.
  f_mean = f_acrosstrial.mean(axis = 0)
C:\Users\Zichen\anaconda3\envs\2ptank\Lib\site-packages\numpy\_core\_methods.py:137: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
C:\Users\Zichen\anaconda3\envs\2ptank\Lib\site-packages\numpy\_core\_methods.py:223: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\Users\Zichen\anaconda3\envs\2ptank\Lib\site-packages\numpy\_core\_methods.py:181: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
C:\Users\Zichen\anaconda3\envs\2ptank\Lib\site-packages\numpy\_core\_methods.py:212: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
C:\Users\Zichen\AppData\Local\Temp\ipykernel_63248\2529180685.py:46: RuntimeWarning: Mean of empty slice.
  f_mean = f_acrosstrial.mean(axis = 0)


DSI > 0.5, angle within +/-45deg
DSI > 0, angle within +/-1000deg
DSI > 0, angle within +/-1000deg
DSI > 0, angle within +/-1000deg
DSI > 0, angle within +/-1000deg
low passs assuming framerate ~2
smoothed
low passs assuming framerate ~2
smoothed
low passs assuming framerate ~2
smoothed
low passs assuming framerate ~2
smoothed
low passs assuming framerate ~2
smoothed
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
baseline: 13 to 20
response: 30 to 32
baselin